# ResearchLanka Full Kaggle Pipeline

Run this notebook in Kaggle to rebuild the dataset from uploaded raw source data, create processed/final datasets, generate embeddings, train classifiers, compare model metrics, and zip outputs for download.

Expected Kaggle input dataset path:

```text
/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data
```

If your dataset path is different, update `DATASET_DATA_DIR` in the setup cell.

## 1. Clone Code And Copy Uploaded Data

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data")

print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)

In [ ]:
%cd /kaggle/working
!rm -rf code
!git clone {REPO_URL} code
%cd /kaggle/working/code/backend
!ls scripts src requirements.txt

In [ ]:
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -50

## 2. Install Dependencies

Kaggle may print dependency conflict warnings. If the install finishes with `Successfully installed`, continue unless a later command fails.

In [ ]:
%cd /kaggle/working/code/backend
!pip install $(grep -v '^psycopg2==' requirements.txt)

## 3. Convert Repository Raw Files To `repositories_combined.csv`

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/processing/map_to_common_schema.py --all
!python scripts/processing/convert_repositories_jsonl_to_csv.py
!ls -lh data/processed/repositories_combined.csv

## 4. Create `sljol.csv` From SLJOL Raw Crossref JSONL

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/processing/map_to_common_schema.py --id sljol
!python scripts/processing/convert_repositories_jsonl_to_csv.py \
  --input data/processed/repositories/sljol.jsonl \
  --output data/raw/sljol/sljol.csv
!ls -lh data/raw/sljol/sljol.csv

## 5. Verify Four Source Inputs For Merge

In [ ]:
!find data -name 'openalex_sri_lanka_works.csv' -o -name 'crossref_clean_2016_2026_enriched.csv' -o -name 'crossref_sri_lanka_works.csv' -o -name 'repositories_combined.csv' -o -name 'sljol.csv'

## 6. Merge Sources And Deduplicate

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/processing/kaggle_merge_common_dataset.py \
  --input-dir data \
  --output-dir data/processed/common
!ls -lh data/processed/common

## 7. Final Cleaning And Analysis-Ready Dataset

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/processing/build_final_common_dataset.py
!python scripts/processing/build_year_filtered_dataset.py --start-year 2016 --end-year 2026
!python scripts/processing/build_language_normalized_dataset.py
!python scripts/processing/build_multivalue_normalized_dataset.py
!python scripts/processing/build_analysis_ready_dataset.py
!ls -lh data/processed/common | tail -30

## 8. Optional Model-Ready Text And TF-IDF Exports

In [ ]:
%cd /kaggle/working/code/backend
!make model-text PYTHON=python
!ls -lh data/processed/common/publication_*_for_model_all_years.csv data/processed/common/publication_tfidf_* 2>/dev/null || true

## 9. Embeddings

In [ ]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python
!ls -lh data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings.parquet data/models/publication_text_embeddings_manifest.json data/models/publication_text_embeddings_summary.txt

## 10. Logistic Regression Classifier

In [ ]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python
!cat data/models/logistic_regression_primary_domain_metrics.txt

## 11. Linear SVM Classifier

This is the heavier model. The settings below are safer than the default full trigrams/50k-feature run. Increase `--max-features` or add more `--c-values` if Kaggle has enough memory/time.

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,keywords \
  --ngram-max 2 \
  --max-features 30000 \
  --c-values 0.1,1,10 \
  --cv-folds 3
!cat data/models/linear_svm_primary_domain_metrics.txt

## 12. Optional Hierarchical Linear SVM

Run this if Kaggle still has enough time and memory.

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_hierarchical.py
!find data -path '*linear_svm*' -type f | sort

## 13. Compare Model Metrics

In [ ]:
%cd /kaggle/working/code/backend
!grep -E "model_family|label_column|accuracy|macro_f1|weighted_f1" data/models/*metrics.txt || true

## 14. Zip Outputs For Download

In [ ]:
%cd /kaggle/working/code/backend
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip

Download this file from the Kaggle output panel:

```text
/kaggle/working/researchlanka-kaggle-outputs.zip
```